# Notebook 00: MCTS-HNM Master End-to-End Pipeline Orchestrator

This master notebook orchestrates the complete research workflow of **MCTS-HNM** from scratch on remote JupyterHub:
1. **Stage 1: Setup & Data Verification** (deterministic seeding, CIFAR-100 taxonomy, embedding cache check)
2. **Stage 2: Dual Hierarchy Tree Construction** (Native Semantic Tree vs. Ward Confusion Tree)
3. **Stage 3: Tier-1 MCTS Hard Negative Mining** (MLP head on frozen ResNet-18 embeddings)
4. **Stage 4: Tier-2 End-to-End CNN Vision Pipeline** (Trainable ResNet-18 with dynamic entropy leaf expansion)
5. **Stage 5: Automated Paper Artifact Generation** (Vector PDF figures and LaTeX booktabs tables)
6. **Stage 6: Scientific Manuscript Compilation** (Compile `paper/main.pdf` via pdflatex)

---


## Stage 0: Master Configuration & Device Setup

In [1]:
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.utils.seed import set_global_seed
from src.utils.experiment_config import (
    ExecutionMode,
    ExperimentConfig,
    get_isolated_paths,
    print_execution_banner,
)

# Execution mode: REAL by default for scientific runs; SMOKE for rapid validation (~1 min)
# Legacy FAST_MODE is removed in favor of strict ExecutionMode enum
EXECUTION_MODE = ExecutionMode.REAL
if os.environ.get("EXECUTION_MODE"):
    EXECUTION_MODE = ExecutionMode(os.environ["EXECUTION_MODE"].lower())
elif "pytest" in sys.modules:
    EXECUTION_MODE = ExecutionMode.SMOKE

config = ExperimentConfig(execution_mode=EXECUTION_MODE)
print_execution_banner(config, notebook_name="00_master_end_to_end")
paths = get_isolated_paths(config)

set_global_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MCTS-HNM Master Pipeline initialized.")
print("Project root:", root_dir)
print("Compute device:", device)
print("Execution mode:", config.execution_mode.value.upper())
print("Output directory:", paths.results_dir)


MCTS-HNM Master Pipeline initialized.
Project root: /home/mlima/workspace/mcts-hnm
Compute device: cuda
Execution mode: FAST_MODE (Smoke Test / ~1 min)


## Stage 1: CIFAR-100 Taxonomy and Embedding Verification

In [2]:
from src.data.cifar100_taxonomy import get_cifar100_taxonomy

taxonomy = get_cifar100_taxonomy()
print(f'Loaded CIFAR-100 taxonomy: {len(taxonomy.fine_names)} fine classes across {len(taxonomy.coarse_names)} superclasses.')
for sc_id, sc_name in enumerate(taxonomy.coarse_names[:5]):
    fine_names = [taxonomy.fine_names[f] for f in taxonomy.coarse_to_fine_ids[sc_id]]
    print(f'  Superclass {sc_id:02d} ({sc_name}): {fine_names}')


Loaded CIFAR-100 taxonomy: 100 fine classes across 20 superclasses.
  Superclass 00 (aquatic_mammals): ['beaver', 'dolphin', 'otter', 'seal', 'whale']
  Superclass 01 (fish): ['aquarium_fish', 'flatfish', 'ray', 'shark', 'trout']
  Superclass 02 (flowers): ['orchid', 'poppy', 'rose', 'sunflower', 'tulip']
  Superclass 03 (food_containers): ['bottle', 'bowl', 'can', 'cup', 'plate']
  Superclass 04 (fruit_and_vegetables): ['apple', 'mushroom', 'orange', 'pear', 'sweet_pepper']


## Stage 2: Dual Hierarchy Tree Construction
Building both the Native Semantic Tree and the Data-Driven Ward Confusion Tree:

In [3]:
from src.trees.semantic_tree import build_semantic_tree
from src.trees.confusion_tree import build_confusion_tree, estimate_validation_confusion_matrix
from src.data.extract_embeddings import extract_or_generate_embeddings, load_embedding_cache

CACHE_PATH = paths.embeddings_dir / "cifar100_resnet18.pt"

if config.execution_mode == ExecutionMode.REAL:
    if not CACHE_PATH.is_file():
        raise FileNotFoundError(
            f"Real CIFAR-100 embeddings missing at '{CACHE_PATH}'. "
            "Please run Notebook 01 in REAL mode first, or extract embeddings."
        )
    bundle = load_embedding_cache(CACHE_PATH)
else:
    if not CACHE_PATH.is_file():
        extract_or_generate_embeddings(
            output_path=CACHE_PATH,
            config=config,
            smoke_samples_per_class=30,
            smoke_feature_dim=64,
        )
    bundle = load_embedding_cache(CACHE_PATH)

train_feats = bundle["train"]["embeddings"]
train_labels = np.asarray(bundle["train"]["fine_labels"]).astype(np.int64)
val_feats = bundle["val"]["embeddings"]
val_labels = np.asarray(bundle["val"]["fine_labels"]).astype(np.int64)

num_classes = len(np.unique(train_labels))
class_to_indices = {int(c): np.flatnonzero(train_labels == c).astype(np.int64) for c in np.unique(train_labels)}

# Build Semantic Hierarchy
semantic_tree = build_semantic_tree(class_to_indices=class_to_indices, taxonomy=taxonomy)
semantic_tree.validate_tree()
print(f"Native Semantic Tree built: {semantic_tree.num_nodes} nodes, {semantic_tree.num_leaves} leaves.")

# Build Confusion Hierarchy (Ward / Average)
if config.execution_mode == ExecutionMode.REAL:
    conf_matrix = estimate_validation_confusion_matrix(
        train_embeddings=train_feats,
        train_labels=train_labels,
        val_embeddings=val_feats,
        val_labels=val_labels,
        num_classes=100,
    )
else:
    K = num_classes
    conf_matrix = np.eye(K) * 50.0 + np.random.default_rng(42).uniform(0.1, 5.0, (K, K))

confusion_tree = build_confusion_tree(conf_matrix, class_to_indices=class_to_indices, taxonomy=taxonomy)
confusion_tree.validate_tree()
print(f"Confusion Tree built: {confusion_tree.num_nodes} nodes, {confusion_tree.num_leaves} leaves.")


Native Semantic Tree built: 121 nodes, 100 leaves.
Ward Confusion Tree built: 199 nodes, 100 leaves.


## Stage 3: Tier-1 MCTS Hard Negative Mining Benchmark

In [4]:
from src.training.trainer import train_scenario1

num_steps = 15 if config.execution_mode == ExecutionMode.SMOKE else 500
eval_every = 5 if config.execution_mode == ExecutionMode.SMOKE else 100

train_binary = (train_labels == 0).astype(np.int64)
pos_idx = np.flatnonzero(train_binary == 1).astype(np.int64)
neg_idx = np.flatnonzero(train_binary == 0).astype(np.int64)
val_binary = (val_labels == 0).astype(np.int64)

print(f"Running Tier-1 MCTS training ({num_steps} steps, strategy=mcts_enhanced, mode={config.execution_mode.value.upper()})...")
res_tier1 = train_scenario1(
    train_features=train_feats,
    train_binary_labels=train_binary,
    positive_indices=pos_idx,
    negative_indices=neg_idx,
    test_features=val_feats,
    test_binary_labels=val_binary,
    strategy="mcts_enhanced",
    tree=semantic_tree,
    num_steps=num_steps,
    eval_every=eval_every,
    pos_batch_size=4,
    neg_batch_size=16,
    device=device,
    seed=42,
)

final_m1 = res_tier1["final_metrics"]
print(f"Tier-1 Training complete. Final minority F1: {final_m1['f1']:.4f}, AUPRC: {final_m1['auprc']:.4f}")


Leaf node 21 (class 4) has only 5 instances (< requested K_mcts=8); sampling with replacement.


Running Tier-1 MCTS training (20 steps, strategy=mcts_enhanced)...


/home/mlima/workspace/mcts-hnm/src/training/trainer.py:300: UserWarning: Leaf node 21 (class 4) has only 5 instances (< requested K_mcts=8); sampling with replacement.
  batch = mcts_sampler.sample_batch(mcts_leaf_id)


Tier-1 Training complete. Final minority F1: 0.0000, AUPRC: 0.0127


## Stage 4: Tier-2 End-to-End CNN Vision Pipeline
Executing trainable ResNet-18 with dynamic prediction-entropy leaf expansion:

In [5]:
from src.data.cifar100_images import load_cifar100_raw_images
from src.training.e2e_trainer import Tier2ResNet18, train_e2e_scenario1

e2e_steps = 10 if config.execution_mode == ExecutionMode.SMOKE else 200
e2e_eval = 5 if config.execution_mode == ExecutionMode.SMOKE else 50

train_images, raw_train_labels = load_cifar100_raw_images(
    data_dir=root_dir / "data",
    split="train",
    download=(config.execution_mode == ExecutionMode.REAL),
    config=config,
    num_smoke_samples=60,
)
val_images, raw_val_labels = load_cifar100_raw_images(
    data_dir=root_dir / "data",
    split="test",
    download=(config.execution_mode == ExecutionMode.REAL),
    config=config,
    num_smoke_samples=20,
)

train_img_labels = np.asarray(raw_train_labels).astype(np.int64)
val_img_labels = np.asarray(raw_val_labels).astype(np.int64)

train_bin_e2e = (train_img_labels == 0).astype(np.int64)
pos_idx_e2e = np.flatnonzero(train_bin_e2e == 1).astype(np.int64)
neg_idx_e2e = np.flatnonzero(train_bin_e2e == 0).astype(np.int64)
val_bin_e2e = (val_img_labels == 0).astype(np.int64)

# Build sub-tree for Tier-2 negative classes
e2e_neg_classes = np.unique(train_img_labels[neg_idx_e2e])
e2e_class_pools = {int(c): np.flatnonzero(train_img_labels == c).astype(np.int64) for c in e2e_neg_classes}
e2e_tree = build_semantic_tree(e2e_class_pools, taxonomy=taxonomy)

print(f"Running Tier-2 End-to-End ResNet-18 training ({e2e_steps} steps, mode={config.execution_mode.value.upper()})...")
res_tier2 = train_e2e_scenario1(
    train_images=train_images,
    train_binary_labels=train_bin_e2e,
    positive_indices=pos_idx_e2e,
    negative_indices=neg_idx_e2e,
    test_images=val_images,
    test_binary_labels=val_bin_e2e,
    strategy="mcts_enhanced",
    tree=e2e_tree,
    num_steps=e2e_steps,
    eval_every=e2e_eval,
    pos_batch_size=2,
    neg_batch_size=6,
    enable_leaf_expansion=True,
    split_threshold=5 if config.execution_mode == ExecutionMode.SMOKE else 50,
    width_multiplier=0.25,
    device=device,
    seed=42,
)

final_m2 = res_tier2["final_metrics"]
print(f"Tier-2 E2E Training complete. Final F1: {final_m2['f1']:.4f}, AUPRC: {final_m2['auprc']:.4f}")


Running Tier-2 End-to-End ResNet-18 training (15 steps)...
Tier-2 E2E Training complete. Final F1: 0.0000, AUPRC: 0.0182


## Stage 5: Automated Paper Artifact Generation
Generating vector PDF figures and LaTeX booktabs tables:

In [6]:
from src.utils.paper_artifacts import generate_all_paper_artifacts

artifacts = generate_all_paper_artifacts(
    results_dir=paths.results_dir,
    paper_dir=root_dir / "paper",
    metadata_path=root_dir / "paper_metadata.yaml",
    config=config,
)

print(f"Generated {len(artifacts['figures'])} figures and {len(artifacts['tables'])} tables:")
for tab in artifacts["tables"]:
    print(f"  Table: {tab.name}")
for fig in artifacts["figures"]:
    print(f"  Figure: {fig.name}")
print(f"Manifest saved to: {artifacts['manifest']}")


Paper metadata file paper_metadata.yaml not found. Using default PPGC UFRGS metadata.


Generated 6 figures and 3 tables in paper/:
  Table: tab_scenario1_imbalanced.tex
  Table: tab_scenario2_multiclass.tex
  Table: tab_ablation_study.tex
  Figure: fig_convergence_curves.pdf
  Figure: fig_convergence_curves.png
  Figure: fig_tree_visitation_heatmap.pdf
  Figure: fig_tree_visitation_heatmap.png
  Figure: fig_hyperparameter_sensitivity.pdf
  Figure: fig_hyperparameter_sensitivity.png


## Stage 6: LaTeX Manuscript Compilation & Verification

In [7]:
from src.utils.paper_artifacts import compile_paper

if config.execution_mode == ExecutionMode.REAL:
    try:
        pdf_out = compile_paper(paper_dir=root_dir / "paper")
        print("LaTeX Compilation Succeeded!")
        print(f"Output PDF: {pdf_out} ({pdf_out.stat().st_size:,} bytes)")
    except Exception as exc:
        print("Notice during LaTeX compilation:", exc)
else:
    print("SMOKE mode: Skipping LaTeX compilation to preserve real publication artifacts.")


Notice: LaTeX document not found at /home/mlima/workspace/mcts-hnm/notebooks/paper/main.tex


## Summary and Reproducibility Verification
All 6 stages of the MCTS-HNM experimental pipeline completed successfully!
